# The Sockeye models of the paper
The from-scratch Sockeye models, which need their own environment (Sockeye 3 pins an old torch):
our small Transformer with its two levers on the web set (upper rows of Table 2), and the 2022
baseline of Wdowiak retrained with his configuration on three training sets (Table 6). The NLLB
experiments are in `sicilian_nmt.ipynb`. Every model and every result is saved on Drive and a
finished one is skipped, so *Run all* only computes what is missing; a T4 is enough.

In [10]:
!pip -q install subword-nmt sacrebleu uv
import os, sys, json
if not os.path.exists('SicilianNMT'):
    !git clone -q https://github.com/dmeoli/SicilianNMT.git
!git -C SicilianNMT pull -q
# Sockeye 3 pins an old torch, so it gets its own environment
if not os.path.exists('/content/sk/bin/sockeye-train'):
    !uv venv -q --python 3.10 /content/sk && uv pip install -q --python /content/sk/bin/python 'sockeye>=3,<4' 'torch==1.13.1'
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/SicilianNMT-colab'
sys.path.append('SicilianNMT/experiments/baseline')
import recipe
print('ready')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ready


## Our from-scratch baseline and its levers (upper rows of Table 2)
The small Transformer of `experiments/baseline/train.sh`, trained for scn->en on the web set and stopped on its 1,000
validation pairs: the plain baseline (raw text, joint BPE of 4000 merges), lever B (Napizia tokenization and
desinence-biased BPE) and lever D (lever B plus a lemma source factor). All are scored on our test set in the
tokenized space, together with the copy-source floor (`experiments/baseline/baseline.py`); results go to
`baseline/results_baseline.json` and finished variants are skipped.

In [11]:
import baseline
DATA = f'{OUT}/data'
for v in ('baseline', 'leverB', 'leverD'):
    baseline.run(v, f'{OUT}/baseline', DATA, sockeye_bin='/content/sk/bin')
res = json.load(open(f'{OUT}/baseline/results_baseline.json'))
print('floor (copy source):', res['floor'])
for v in ('baseline', 'leverB', 'leverD'):
    print(f"{v:9s} BLEU/chrF (tokenized) {res.get(v)}")

baseline [8.51, 32.73]
leverB [10.94, 34.66]
leverD [10.18, 34.15]
floor (copy source): {'raw': [5.44, 25.42], 'tokenized': [5.18, 25.95]}
baseline  BLEU/chrF (tokenized) [8.51, 32.73]
leverB    BLEU/chrF (tokenized) [10.94, 34.66]
leverD    BLEU/chrF (tokenized) [10.18, 34.15]


## Retraining the 2022 baseline of Wdowiak (Table 6)

The published baseline of Wdowiak (25.1 BLEU en->scn, 29.1 scn->en) is a small Sockeye
Transformer trained on sentences he aligned by hand, scored on his own hand-selected test (the
121 trilingual lines from AS38-39, the `validation` sheet of his data) in the space of the
Napizia tokenizer. Here the same model, with his configuration, is trained on three sets that
differ only in where the parallel text comes from (built by `experiments/baseline/recipe_data.py`,
private, on Drive under `recipe/`):

- **A**: his hand-aligned sheets, which checks that our re-implementation reproduces his score;
- **B**: our automatic alignment of the born-digital *Arba Sicula* issues, plus the same textbook
  exercises (without Dieli's translations of Pitrè, which his sheet mixes with the journal, so B
  has less data than A);
- **C**: A and B together.

Each set is guarded against his test and against the 500 validation lines that stop the
training. The code is `experiments/baseline/recipe.py`; its docstring lists his configuration
and the two deliberate differences (we stop on our validation set, not on his test, and we train
from scratch). A T4 is enough; every model and every result is saved on Drive, so a
disconnected run resumes where it stopped.

In [12]:
# A (his data), B (ours), C (both), in both directions; skips what is already done
for name in ('A', 'B', 'C'):
    for s, t in (('en', 'scn'), ('scn', 'en')):
        recipe.run(name, f'{OUT}/recipe', s, t, sockeye_bin='/content/sk/bin')
res = json.load(open(f'{OUT}/recipe/results_recipe.json'))
print(f"{'set':6s} {'en->scn':>8s} {'scn->en':>8s}   (Wdowiak 2022: 25.1 / 29.1)")
for name in ('A', 'B', 'C'):
    print(f"{name:6s} {res.get(f'{name} en->scn', '-'):>8} {res.get(f'{name} scn->en', '-'):>8}")

A en->scn 15.97
A scn->en 16.65
B en->scn 13.2
B scn->en 13.14
C en->scn 22.45
C scn->en 20.69
set     en->scn  scn->en   (Wdowiak 2022: 25.1 / 29.1)
A         15.97    16.65
B          13.2    13.14
C         22.45    20.69
